# Error Analysis: FP & FN Investigation
Uses cached inference outputs. High-confidence FPs = likely missing annotations. FNs = phantom annotations or hard cases.

In [1]:
%matplotlib inline

import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore', category=UserWarning)

import torch, numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib.patches as patches
import sklearn.neighbors
from pathlib import Path
from PIL import Image

import animaloc
from animaloc.eval.lmds import HerdNetLMDS

DATA_ROOT = Path('..').resolve()

# ── CHOOSE DATASET ──────────────────────────────────────────
# Option A: FMO03 crops (512x512, previous experiments)
# VAL_DIR = DATA_ROOT / 'data_FMO03_02_05/val/crops_512'
# VAL_CSV = DATA_ROOT / 'data_FMO03_02_05/val/crops_512/herdnet_format.csv'
# PTH = DATA_ROOT / 'output/fmo03_convnext_opt/2026-04-06/21-48-42/best_model.pth'

# Option B: iguana_hn patches (new dataset)
VAL_DIR = DATA_ROOT / 'data_iguana_hn/val_patches'
VAL_CSV = DATA_ROOT / 'data_iguana_hn/val_patches/herdnet_format_with_hn.csv'
PTH = DATA_ROOT / 'output/hn_full_hybrid/2026-04-07/19-07-46/best_model.pth'

print(f'Val dir:  {VAL_DIR}')
print(f'Val CSV:  {VAL_CSV}')
print(f'Model:    {PTH}')


Val dir:  /home/christian/hnee/HerdNet/data_iguana_hn/val_patches
Val CSV:  /home/christian/hnee/HerdNet/data_iguana_hn/val_patches/herdnet_format_with_hn.csv
Model:    /home/christian/hnee/HerdNet/output/hn_full_hybrid/2026-04-07/19-07-46/best_model.pth


In [2]:
# Config
ADAPT_TS = 0.30
KERNEL_SIZE = (3, 3)
NEG_TS = 0.05
MATCH_RADIUS = 25  # in heatmap space

# ── Cache setup ──
import hashlib
cache_key = hashlib.md5(f'{PTH.stat().st_size}_{PTH.stat().st_mtime_ns}_{VAL_CSV}'.encode()).hexdigest()[:12]
CACHE_DIR = DATA_ROOT / 'output/hparam_search/cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
cache_file = CACHE_DIR / f'{cache_key}.pt'

image_names = pd.read_csv(str(VAL_CSV))['images'].unique().tolist()

if cache_file.exists():
    print(f'Loading from cache: {cache_file.name}')
    cached = torch.load(str(cache_file), weights_only=False)
    DOWN_RATIO = 512 // cached[0]['heatmap'].shape[2]
else:
    print('Running inference (will be cached for next time)...')
    # Load model
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    checkpoint = torch.load(str(PTH), map_location='cpu', weights_only=False)
    cfg = checkpoint['config']
    model_name = cfg['model']['name']
    model_kwargs = cfg['model'].get('kwargs', {})
    model = animaloc.models.__dict__[model_name](**model_kwargs)
    model.reshape_classes(cfg['datasets']['num_classes'])
    state_dict = {k.replace('model.', '', 1) if k.startswith('model.') else k: v
                  for k, v in checkpoint['model_state_dict'].items()}
    model.load_state_dict(state_dict, strict=False)
    model = model.to(device).eval()
    DOWN_RATIO = cfg['model']['kwargs'].get('down_ratio', 4)
    print(f'Model: {model_name}, down_ratio={DOWN_RATIO}')

    import albumentations
    val_df = pd.read_csv(str(VAL_CSV))
    end_transforms = animaloc.data.transforms.DownSample(down_ratio=DOWN_RATIO, anno_type='point')
    val_dataset = animaloc.datasets.CSVDataset(
        csv_file=val_df, root_dir=str(VAL_DIR),
        albu_transforms=[albumentations.Normalize()],
        end_transforms=[end_transforms],
    )
    dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=1, shuffle=False)

    cached = []
    with torch.no_grad():
        for idx, (images, targets) in enumerate(dataloader):
            images_gpu = images.to(device)
            with torch.amp.autocast('cuda', enabled=device.type == 'cuda'):
                heatmap, clsmap = model(images_gpu)
            gt_points = targets['points'].squeeze(0).tolist()
            cached.append({
                'heatmap': heatmap.cpu(), 'clsmap': clsmap.cpu(),
                'gt_coords': [(p[1], p[0]) for p in gt_points],
                'gt_labels': targets['labels'].squeeze(0).tolist(),
            })
    torch.save(cached, cache_file)
    del model; torch.cuda.empty_cache()
    print(f'Cached to {cache_file.name}')

# Run LMDS on cached outputs
heatmap_size = cached[0]['heatmap'].shape[2]
sf = heatmap_size // cached[0]['clsmap'].shape[2]
lmds = HerdNetLMDS(kernel_size=KERNEL_SIZE, adapt_ts=ADAPT_TS, neg_ts=NEG_TS, up=True, scale_factor=sf)

results = []
for idx, sample in enumerate(cached):
    counts, locs, labels, scores, dscores = lmds((sample['heatmap'], sample['clsmap']))
    results.append({
        'idx': idx,
        'image_name': image_names[idx] if idx < len(image_names) else f'img_{idx}.jpg',
        'gt_coords': sample['gt_coords'],
        'gt_labels': sample['gt_labels'],
        'pred_locs': locs[0],
        'pred_scores': dscores[0],
        'heatmap': sample['heatmap'][0, 0].numpy(),
        'gt_count': len([l for l in sample['gt_labels'] if l == 1]),
        'pred_count': len(locs[0]),
    })

print(f'{len(results)} images | GT: {sum(r["gt_count"] for r in results)} | Pred: {sum(r["pred_count"] for r in results)}')


FileNotFoundError: [Errno 2] No such file or directory: '/home/christian/hnee/HerdNet/data_iguana_hn/val_patches/herdnet_format_with_hn.csv'

In [ ]:
def match_preds(gt_coords, pred_coords, pred_scores, radius=MATCH_RADIUS):
    if not gt_coords and not pred_coords: return [], [], []
    if not pred_coords: return [], [], [(i, gt) for i, gt in enumerate(gt_coords)]
    if not gt_coords: return [], [(i, pred_coords[i], pred_scores[i]) for i in range(len(pred_coords))], []
    nn = sklearn.neighbors.NearestNeighbors(n_neighbors=1, metric='euclidean').fit(pred_coords)
    dists, idxs = nn.kneighbors(gt_coords)
    matches = sorted([(k, dists[k,0], idxs[k,0]) for k in range(len(gt_coords))], key=lambda x: x[1])
    used_gt, used_pred, tp = set(), set(), []
    for gi, d, pi in matches:
        if gi in used_gt or pi in used_pred: continue
        if d <= radius:
            tp.append((gi, pi, d, pred_scores[pi] if pi < len(pred_scores) else 0))
            used_gt.add(gi); used_pred.add(pi)
    fp = [(i, pred_coords[i], pred_scores[i]) for i in range(len(pred_coords)) if i not in used_pred]
    fn = [(i, gt_coords[i]) for i in range(len(gt_coords)) if i not in used_gt]
    return tp, fp, fn

all_tp, all_fp, all_fn = [], [], []
for r in results:
    # Filter to only real annotations (label==1), skip hard-negative dummy points
    real_gt = [(c, l) for c, l in zip(r['gt_coords'], r['gt_labels']) if l == 1]
    gt_coords_real = [c for c, l in real_gt]
    tp, fp, fn = match_preds(gt_coords_real, r['pred_locs'], r['pred_scores'])
    for gi, pi, d, s in tp:
        all_tp.append((r['idx'], r['image_name'], gt_coords_real[gi], r['pred_locs'][pi], d, s))
    for pi, c, s in fp:
        all_fp.append((r['idx'], r['image_name'], c, s))
    for gi, c in fn:
        all_fn.append((r['idx'], r['image_name'], c))

print(f'TP={len(all_tp)} FP={len(all_fp)} FN={len(all_fn)}')
print(f'Recall={len(all_tp)/(len(all_tp)+len(all_fn)):.3f} Precision={len(all_tp)/(len(all_tp)+len(all_fp)):.3f}')


## Full-Tile View
Reconstruct detections and errors in original full-image coordinates.
Groups crops from the same drone image and shows FP/FN in the full context.

In [ ]:
import re

# Reconstruct full-image coordinates from crop offsets
# Crop name: {BASE}_x{OFFSET_X}_y{OFFSET_Y}.jpg
def parse_crop_name(name):
    m = re.match(r'(.+)_x(\d+)_y(\d+)\.jpg', name)
    if m:
        return m.group(1), int(m.group(2)), int(m.group(3))
    return name, 0, 0

# Build full-image error map
full_image_errors = {}  # base_name -> {gt: [], pred: [], fp: [], fn: [], tp: []}

for r in results:
    base, ox, oy = parse_crop_name(r['image_name'])
    if base not in full_image_errors:
        full_image_errors[base] = {'gt': [], 'pred': [], 'fp': [], 'fn': [], 'tp': []}
    
    # Map GT and pred to full-image coords
    for (gy, gx), lbl in zip(r['gt_coords'], r['gt_labels']):
        if lbl == 1:
            full_image_errors[base]['gt'].append((gy * DOWN_RATIO + oy, gx * DOWN_RATIO + ox))
    for (py, px), score in zip(r['pred_locs'], r['pred_scores']):
        full_image_errors[base]['pred'].append((py * DOWN_RATIO + oy, px * DOWN_RATIO + ox, score))

# Map FP/FN to full-image coords
for img_idx, img_name, coord, score in all_fp:
    base, ox, oy = parse_crop_name(img_name)
    full_y = coord[0] * DOWN_RATIO + oy
    full_x = coord[1] * DOWN_RATIO + ox
    full_image_errors[base]['fp'].append((full_y, full_x, score))

for img_idx, img_name, coord in all_fn:
    base, ox, oy = parse_crop_name(img_name)
    full_y = coord[0] * DOWN_RATIO + oy
    full_x = coord[1] * DOWN_RATIO + ox
    full_image_errors[base]['fn'].append((full_y, full_x))

# Deduplicate FN in full-image space (same iguana from overlapping crops)
for base, data in full_image_errors.items():
    unique_fn = []
    for fy, fx in data['fn']:
        if not any(abs(fy-uy) < 40 and abs(fx-ux) < 40 for uy, ux in unique_fn):
            unique_fn.append((fy, fx))
    data['fn_unique'] = unique_fn
    # Same for FP
    unique_fp = []
    for fy, fx, s in data['fp']:
        if not any(abs(fy-uy) < 40 and abs(fx-ux) < 40 for uy, ux, _ in unique_fp):
            unique_fp.append((fy, fx, s))
    data['fp_unique'] = unique_fp

print(f'Full images: {len(full_image_errors)}')
for base, data in sorted(full_image_errors.items()):
    n_gt = len(data['gt'])
    n_fp = len(data['fp_unique'])
    n_fn = len(data['fn_unique'])
    status = '  ' if n_fp == 0 and n_fn == 0 else ' *'
    print(f'{status} {base}: GT={n_gt:>3}  FP={n_fp}  FN={n_fn}')


In [ ]:
# Export FP and FN locations as CSV for correction workflow
fp_rows = []
fn_rows = []

for base, data in full_image_errors.items():
    for fy, fx, score in data['fp_unique']:
        fp_rows.append({'base_image': base, 'x': int(fx), 'y': int(fy), 'score': round(score, 4), 'type': 'false_positive'})
    for fy, fx in data['fn_unique']:
        fn_rows.append({'base_image': base, 'x': int(fx), 'y': int(fy), 'type': 'false_negative'})

fp_df = pd.DataFrame(fp_rows)
fn_df_export = pd.DataFrame(fn_rows)
errors_df = pd.concat([fp_df, fn_df_export], ignore_index=True)

out_path = DATA_ROOT / 'output' / 'hparam_search' / 'detection_errors.csv'
errors_df.to_csv(out_path, index=False)
print(f'Exported {len(fp_rows)} FP + {len(fn_rows)} FN to {out_path}')
print(f'\nFP summary (for Label Studio upload):')
print(fp_df.to_string(index=False))


In [ ]:
def show_error(r, coord_hm, error_type='FP', score=None, crop_size=100):
    """coord_hm is in heatmap space — scale to image space for display."""
    img_path = VAL_DIR / r['image_name']
    if not img_path.exists():
        print(f'  Not found: {img_path}'); return
    img = np.array(Image.open(img_path))
    h, w = img.shape[:2]
    # Scale heatmap coords to image space
    cy, cx = int(coord_hm[0] * DOWN_RATIO), int(coord_hm[1] * DOWN_RATIO)
    y1, y2 = max(0, cy-crop_size), min(h, cy+crop_size)
    x1, x2 = max(0, cx-crop_size), min(w, cx+crop_size)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].imshow(img)
    axes[0].add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1, lw=2, ec='yellow', fc='none'))
    axes[0].set_title(r['image_name'], fontsize=8); axes[0].axis('off')
    
    axes[1].imshow(img[y1:y2, x1:x2])
    for gy, gx in r['gt_coords']:  # heatmap space
        gy_img, gx_img = gy*DOWN_RATIO, gx*DOWN_RATIO
        if y1<=gy_img<=y2 and x1<=gx_img<=x2:
            axes[1].plot(gx_img-x1, gy_img-y1, 'o', color='lime', ms=14, mfc='none', mew=2)
    for (py,px), s in zip(r['pred_locs'], r['pred_scores']):  # heatmap space
        py_img, px_img = py*DOWN_RATIO, px*DOWN_RATIO
        if y1<=py_img<=y2 and x1<=px_img<=x2:
            axes[1].plot(px_img-x1, py_img-y1, 'x', color='red', ms=10, mew=2)
            axes[1].annotate(f'{s:.2f}', (px_img-x1+5, py_img-y1-5), color='red', fontsize=8)
    color = 'yellow' if error_type=='FP' else 'cyan'
    axes[1].add_patch(patches.Rectangle((cx-x1-15, cy-y1-15), 30, 30, lw=3, ec=color, fc='none'))
    axes[1].set_title(f'{error_type} | green=GT red=pred', fontsize=9); axes[1].axis('off')
    
    hm = r['heatmap']
    hy1,hx1 = y1//DOWN_RATIO, x1//DOWN_RATIO
    hy2,hx2 = y2//DOWN_RATIO, x2//DOWN_RATIO
    hm_crop = hm[hy1:hy2, hx1:hx2]
    axes[2].imshow(hm_crop, cmap='hot', vmin=0, vmax=max(0.3, hm_crop.max()))
    axes[2].set_title(f'Heatmap max={hm_crop.max():.3f}', fontsize=9); axes[2].axis('off')
    
    t = f'{error_type}'
    if score is not None: t += f' score={score:.4f}'
    fig.suptitle(t, fontsize=11, fontweight='bold')
    plt.tight_layout()
    from IPython.display import display
    display(fig)
    plt.close(fig)


## High-Confidence False Positives
If you see an iguana → **missing annotation**

In [ ]:
fp_sorted = sorted(all_fp, key=lambda x: x[3], reverse=True)
print(f'Total FPs: {len(fp_sorted)}')
for i, (idx, name, coord, score) in enumerate(fp_sorted[:20]):
    print(f'  {i+1:>2}. score={score:.4f}  {name}  ({coord[0]:.0f},{coord[1]:.0f})')

In [ ]:
for i, (idx, name, coord, score) in enumerate(fp_sorted[:10]):
    show_error(results[idx], coord, 'FP', score)

## Threshold Comparison: adapt_ts=0.60 vs 0.50
What extra detections do we gain by lowering the threshold?

In [ ]:
# Run LMDS at both thresholds and find the EXTRA detections
sf = cached[0]['heatmap'].shape[2] // cached[0]['clsmap'].shape[2]
lmds_strict = HerdNetLMDS(kernel_size=(3,3), adapt_ts=0.60, neg_ts=0.05, up=True, scale_factor=sf)
lmds_relaxed = HerdNetLMDS(kernel_size=(3,3), adapt_ts=0.50, neg_ts=0.05, up=True, scale_factor=sf)

extra_dets = []  # detections found at 0.50 but NOT at 0.60
for idx, sample in enumerate(cached):
    _, locs_s, _, _, dscores_s = lmds_strict((sample['heatmap'], sample['clsmap']))
    _, locs_r, _, _, dscores_r = lmds_relaxed((sample['heatmap'], sample['clsmap']))
    
    strict_set = set((round(y,1), round(x,1)) for y, x in locs_s[0])
    
    for j, (loc, score) in enumerate(zip(locs_r[0], dscores_r[0])):
        key = (round(loc[0],1), round(loc[1],1))
        if key not in strict_set:
            img_name = image_names[idx] if idx < len(image_names) else f'img_{idx}'
            extra_dets.append((idx, img_name, loc, score))

extra_dets.sort(key=lambda x: x[3], reverse=True)
print(f'Extra detections at adapt_ts=0.50 vs 0.60: {len(extra_dets)}')
print(f'\n{"#":<4} {"Score":<8} {"Image":<45} {"(y,x)"}')
print('-' * 75)
for i, (idx, name, coord, score) in enumerate(extra_dets):
    print(f'{i+1:<4} {score:<8.4f} {name:<45} ({coord[0]:.0f},{coord[1]:.0f})')


In [ ]:
# Visualize each extra detection
# These need to be checked: real iguana = lower threshold is better
for i, (idx, name, coord, score) in enumerate(extra_dets):
    show_error(results[idx], coord, f'NEW at ts=0.50', score, crop_size=80)


### Also check: adapt_ts=0.40 extras
What do we gain going from 0.50 to 0.40?

In [ ]:
lmds_040 = HerdNetLMDS(kernel_size=(3,3), adapt_ts=0.40, neg_ts=0.05, up=True, scale_factor=sf)

extra_040 = []
for idx, sample in enumerate(cached):
    _, locs_r, _, _, dscores_r = lmds_relaxed((sample['heatmap'], sample['clsmap']))
    _, locs_040, _, _, dscores_040 = lmds_040((sample['heatmap'], sample['clsmap']))
    
    relaxed_set = set((round(y,1), round(x,1)) for y, x in locs_r[0])
    
    for j, (loc, score) in enumerate(zip(locs_040[0], dscores_040[0])):
        key = (round(loc[0],1), round(loc[1],1))
        if key not in relaxed_set:
            img_name = image_names[idx] if idx < len(image_names) else f'img_{idx}'
            extra_040.append((idx, img_name, loc, score))

extra_040.sort(key=lambda x: x[3], reverse=True)
print(f'Extra detections at adapt_ts=0.40 vs 0.50: {len(extra_040)}')
print(f'\nShowing top 15 by score:')
for i, (idx, name, coord, score) in enumerate(extra_040[:15]):
    print(f'  {i+1:>2}. score={score:.4f}  {name}  ({coord[0]:.0f},{coord[1]:.0f})')

for i, (idx, name, coord, score) in enumerate(extra_040[:15]):
    show_error(results[idx], coord, f'NEW at ts=0.40', score, crop_size=80)


## False Negatives
If you **don't** see an iguana → **phantom annotation**

In [ ]:
print(f'Total FNs: {len(all_fn)}')
for i, (idx, name, coord) in enumerate(all_fn):
    print(f'  {i+1:>2}. {name}  ({coord[0]:.0f},{coord[1]:.0f})')

In [ ]:
for i, (idx, name, coord) in enumerate(all_fn[:15]):
    show_error(results[idx], coord, 'FN')

## FN Category Analysis
Breaking down false negatives by root cause

In [ ]:
import re

# Categorize all FNs
fn_categories = []
for img_idx, img_name, coord in all_fn:
    r = results[img_idx]
    hm = r['heatmap']
    gy, gx = coord
    HM = hm.shape[0]
    yi = min(max(int(round(gy)), 0), HM-1)
    xi = min(max(int(round(gx)), 0), HM-1)
    y1, y2 = max(0, yi-2), min(HM, yi+3)
    x1, x2 = max(0, xi-2), min(HM, xi+3)
    hm_max = float(hm[y1:y2, x1:x2].max())
    at_edge = yi <= 2 or yi >= HM-3 or xi <= 2 or xi >= HM-3
    
    if at_edge:
        cat = 'edge'
    elif hm_max > 0.1:
        cat = 'below_threshold'
    else:
        cat = 'no_response'
    
    fn_categories.append({
        'img_idx': img_idx, 'img': img_name, 'coord': coord,
        'gy': gy, 'gx': gx, 'hm_max': hm_max, 'category': cat,
    })

# Deduplicate — same iguana in overlapping crops
for entry in fn_categories:
    m = re.match(r'(.+)_x(\d+)_y(\d+)\.jpg', entry['img'])
    if m:
        base, cx, cy = m.group(1), int(m.group(2)), int(m.group(3))
        entry['orig_y'] = int(entry['gy'] * DOWN_RATIO + cy)
        entry['orig_x'] = int(entry['gx'] * DOWN_RATIO + cx)
        entry['base_img'] = base
    else:
        entry['orig_y'] = int(entry['gy'] * DOWN_RATIO)
        entry['orig_x'] = int(entry['gx'] * DOWN_RATIO)
        entry['base_img'] = entry['img']

fn_df = pd.DataFrame(fn_categories)
print(f'Total FNs: {len(fn_df)}')
print(f'  Edge (crop boundary):     {(fn_df["category"]=="edge").sum()}')
print(f'  Below threshold (0.1-0.6): {(fn_df["category"]=="below_threshold").sum()}')
print(f'  No response (<0.1):        {(fn_df["category"]=="no_response").sum()}')

# Deduplicate
seen = {}
fn_df['is_duplicate'] = False
for i, row in fn_df.iterrows():
    key = f"{row['base_img']}_{row['orig_x']//40}_{row['orig_y']//40}"
    if key in seen:
        fn_df.at[i, 'is_duplicate'] = True
    else:
        seen[key] = i
print(f'\nDuplicates (same iguana in overlapping crops): {fn_df["is_duplicate"].sum()}')
print(f'Unique missed iguanas: {(~fn_df["is_duplicate"]).sum()}')


### Edge FNs — iguanas at crop boundaries

In [ ]:
edge_fns = fn_df[fn_df['category'] == 'edge']
print(f'{len(edge_fns)} edge FNs')
for _, row in edge_fns.iterrows():
    print(f'  {row["img"]:45s} ({row["gy"]:.0f},{row["gx"]:.0f}) hm={row["hm_max"]:.3f}')
    show_error(results[row['img_idx']], row['coord'], 'FN (edge)', crop_size=80)


### Below-Threshold FNs — model sees something but not confident enough
Heatmap response 0.1–0.6 — would be detected with a lower `adapt_ts`

In [ ]:
thresh_fns = fn_df[fn_df['category'] == 'below_threshold'].sort_values('hm_max', ascending=False)
print(f'{len(thresh_fns)} below-threshold FNs')
for _, row in thresh_fns.iterrows():
    dup = ' [DUP]' if row['is_duplicate'] else ''
    print(f'  {row["img"]:45s} ({row["gy"]:.0f},{row["gx"]:.0f}) hm={row["hm_max"]:.3f}{dup}')

# Show only unique ones
for _, row in thresh_fns[~thresh_fns['is_duplicate']].iterrows():
    show_error(results[row['img_idx']], row['coord'], f'FN (hm={row["hm_max"]:.2f})', crop_size=80)


### No-Response FNs — model completely blind
Heatmap < 0.1 — either very camouflaged, or **phantom annotations** (check if iguana is actually there)

In [ ]:
blind_fns = fn_df[fn_df['category'] == 'no_response']
print(f'{len(blind_fns)} no-response FNs ({(blind_fns["is_duplicate"]).sum()} duplicates)')

# Show only unique ones
unique_blind = blind_fns[~blind_fns['is_duplicate']]
print(f'Showing {len(unique_blind)} unique:')
for _, row in unique_blind.iterrows():
    print(f'  {row["img"]:45s} ({row["gy"]:.0f},{row["gx"]:.0f}) hm={row["hm_max"]:.4f}')
    show_error(results[row['img_idx']], row['coord'], f'FN (hm={row["hm_max"]:.3f})', crop_size=80)


### Summary: what to do with each category

In [ ]:
unique = fn_df[~fn_df['is_duplicate']]
print('FN Root Cause Analysis')
print('=' * 60)
print(f'  Edge (crop boundary):      {(unique["category"]=="edge").sum():>3}  -> Handled by stitcher at inference')
print(f'  Below threshold:           {(unique["category"]=="below_threshold").sum():>3}  -> Lower adapt_ts or retrain')
print(f'  No response (check GT!):   {(unique["category"]=="no_response").sum():>3}  -> Likely phantom annotations')
print(f'  {"":40s} ----')
print(f'  Total unique missed:       {len(unique):>3}  (of {sum(r["gt_count"] for r in results)} total GT)')
print(f'\nIf no-response FNs are phantom annotations:')
n_phantom = (unique['category']=='no_response').sum()
corrected_gt = sum(r['gt_count'] for r in results) - n_phantom
corrected_tp = len(all_tp)
corrected_r = corrected_tp / corrected_gt if corrected_gt > 0 else 0
print(f'  Corrected GT count: {corrected_gt}')
print(f'  Corrected recall:   {corrected_r:.3f}')


## Score Distribution & PR Curve

In [ ]:
tp_scores = [s for _,_,_,_,_,s in all_tp]
fp_scores = [s for _,_,_,s in all_fp]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.hist(tp_scores, bins=30, alpha=0.7, label=f'TP ({len(tp_scores)})', color='green')
if fp_scores: ax1.hist(fp_scores, bins=30, alpha=0.7, label=f'FP ({len(fp_scores)})', color='red')
ax1.set_xlabel('Score'); ax1.set_ylabel('Count'); ax1.legend(); ax1.set_title('TP vs FP scores')

ts = np.arange(0, 1.01, 0.02)
pr = [(sum(s>=t for s in tp_scores)/(sum(s>=t for s in tp_scores)+sum(s>=t for s in fp_scores)) if (sum(s>=t for s in tp_scores)+sum(s>=t for s in fp_scores))>0 else 1,
       sum(s>=t for s in tp_scores)/(sum(s>=t for s in tp_scores)+len(all_fn)+sum(s<t for s in tp_scores)) if (sum(s>=t for s in tp_scores)+len(all_fn)+sum(s<t for s in tp_scores))>0 else 0) for t in ts]
ax2.plot([r for _,r in pr], [p for p,_ in pr], 'b-', lw=2)
ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision'); ax2.set_title('PR Curve')
ax2.set_xlim(0,1.05); ax2.set_ylim(0,1.05); ax2.grid(alpha=0.3)
plt.tight_layout()
from IPython.display import display
display(fig)
plt.close(fig)

if fp_scores: print(f'FP: min={min(fp_scores):.4f} max={max(fp_scores):.4f} median={np.median(fp_scores):.4f}')
print(f'TP: min={min(tp_scores):.4f} max={max(tp_scores):.4f} median={np.median(tp_scores):.4f}')

## Summary

In [ ]:
n_hc = sum(1 for _,_,_,s in all_fp if s > 0.5)
n_lc = len(all_fp) - n_hc
print(f'FPs: {len(all_fp)} total ({n_hc} high-conf >0.5, {n_lc} low-conf)')
print(f'FNs: {len(all_fn)} total')
print(f'\nIf high-conf FPs are missing labels:')
ct = len(all_tp)+n_hc; cf = n_lc; cn = len(all_fn)
cp = ct/(ct+cf) if ct+cf>0 else 0; cr = ct/(ct+cn) if ct+cn>0 else 0
print(f'  R={cr:.3f} P={cp:.3f} F1={2*cp*cr/(cp+cr) if cp+cr>0 else 0:.3f}')